# Лабораторная работа: Генерация лиц GAN

Запустите ячейки сверху вниз. Полученные метрики внесите в вывод.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)
print('GPU:',tf.config.list_physical_devices('GPU'))

In [ ]:
from google.colab import files
from pathlib import Path
import io,zipfile,tempfile,os
u=files.upload();name=next(n for n in u if n.endswith('.zip'))
root=tempfile.mkdtemp()
with zipfile.ZipFile(io.BytesIO(u[name])) as z:
 for member in z.infolist():
  dest=os.path.realpath(os.path.join(root,member.filename))
  assert dest.startswith(os.path.realpath(root)+os.sep) or dest==os.path.realpath(root)
 z.extractall(root)
paths=[str(p) for p in Path(root).rglob('*') if p.suffix.lower() in ('.jpg','.jpeg','.png')]
assert len(paths)>100,'Нужен ZIP с фотографиями CelebA'
def load(path):
 im=tf.io.decode_image(tf.io.read_file(path),channels=3,expand_animations=False)
 im=tf.image.resize(im,(64,64))/255.;im.set_shape((64,64,3));return im
ds=tf.data.Dataset.from_tensor_slices(paths).shuffle(len(paths)).map(load,num_parallel_calls=tf.data.AUTOTUNE).batch(64,drop_remainder=True).prefetch(tf.data.AUTOTUNE)
latent=128
gen=keras.Sequential([layers.Input((latent,)),layers.Dense(8*8*256),layers.Reshape((8,8,256)),layers.Conv2DTranspose(128,4,strides=2,padding='same',activation='relu'),layers.Conv2DTranspose(64,4,strides=2,padding='same',activation='relu'),layers.Conv2DTranspose(3,4,strides=2,padding='same',activation='sigmoid')])
disc=keras.Sequential([layers.Input((64,64,3)),layers.Conv2D(64,4,strides=2,padding='same'),layers.LeakyReLU(.2),layers.Flatten(),layers.Dense(1)])
class GAN(keras.Model):
 def __init__(self,g,d):
  super().__init__();self.g=g;self.d=d;self.dm=keras.metrics.Mean(name='d_loss');self.gm=keras.metrics.Mean(name='g_loss')
 @property
 def metrics(self):return [self.dm,self.gm]
 def compile(self):
  super().compile();self.do=keras.optimizers.Adam(1e-4);self.go=keras.optimizers.Adam(1e-4);self.loss=keras.losses.BinaryCrossentropy(from_logits=True)
 def train_step(self,real):
  z=tf.random.normal((tf.shape(real)[0],latent));fake=self.g(z,training=True)
  with tf.GradientTape() as t:
   r=self.d(real,training=True);f=self.d(fake,training=True)
   dl=self.loss(tf.ones_like(r),r)+self.loss(tf.zeros_like(f),f)
  self.do.apply_gradients(zip(t.gradient(dl,self.d.trainable_variables),self.d.trainable_variables))
  with tf.GradientTape() as t:
   fake=self.g(tf.random.normal((tf.shape(real)[0],latent)),training=True)
   f=self.d(fake,training=False);gl=self.loss(tf.ones_like(f),f)
  self.go.apply_gradients(zip(t.gradient(gl,self.g.trainable_variables),self.g.trainable_variables))
  self.dm.update_state(dl);self.gm.update_state(gl)
  return {'d_loss':self.dm.result(),'g_loss':self.gm.result()}
gan=GAN(gen,disc);gan.compile()
history=gan.fit(ds,epochs=100)
pd.DataFrame(history.history).plot(grid=True);plt.show()
samples=gen(tf.random.normal((8,latent)),training=False).numpy()
fig,ax=plt.subplots(2,4,figsize=(8,4))
for a,im in zip(ax.flat,samples):a.imshow(np.clip(im,0,1));a.axis('off')
plt.show()
gen.save('generator.keras')